#### Handling sequences with PyTorch

Sequential data
- Ordered in time or space
- Order of the data points contains dependencies between them
- Examples of sequential data:
    - Time series
    - Text
    - Audio waves

In [1]:
import pandas as pd
import torch
import matplotlib.pyplot as mlt
import numpy as np


ELectricity consumption prediction
- Task: predict future electricity consumption based on past patterns
- Electricity consumption dataset:

In [2]:
df = pd.read_csv('electricity_consump/electricity_train.csv')
df.head()

,timestamp,consumption
0,2011-01-01 00:15:00,-0.704319
1,2011-01-01 00:30:00,-0.704319
2,2011-01-01 00:45:00,-0.678983
3,2011-01-01 01:00:00,-0.653647
4,2011-01-01 01:15:00,-0.704319


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105215 entries, 0 to 105214
Data columns (total 2 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   timestamp    105215 non-null  object 
 1   consumption  105215 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.6+ MB


In [4]:
df.describe()

,consumption
count,105215.000000
mean,-0.007469
std,1.056835
min,-1.414483
25%,-0.957931
50%,-0.349363
75%,0.797593
max,3.028924


Train-test split
- No random splitting for time series!
- Look-ahead bias: model has info about the future
- Solution: split by time

Creating Sequences
- Sequence length = number of data points in one training example
    - 24 x 4 = 96 -> consider last 24 hours
- Predict single next data point

Creating sequences in Python
- Take data and sequence length as inputs
- Initialize imputs and targets lists
- Iterate over data points
- Define inputs and target
- Append to pre-initialized lists
- Return inputs and targets as NumPy arrays

In [16]:
import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    for i in range(len(df) - seq_length):
        x = df.iloc[i: (i+seq_length), 1]
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

### TensorDataset
- Create training examples

In [17]:
seq_length = 24*4
X_train, y_train = create_sequences(df, seq_length)
print(X_train.shape,  y_train.shape)

(105119, 96) (105119,)


Convert them to a Torch Dataset

In [14]:
from torch.utils.data import TensorDataset

dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)

#### Applicability to other sequential data
Same techniques are applicable to other sequences:
- Large Language Models
- Speech recognition

Exercise:
- Generating sequences

In [ ]:
import numpy as np

def create_sequences(df, seq_length):
    xs, ys = [], []
    # Iterate over data indices
    for i in range(len(df) - seq_length):
      	# Define inputs
        x = df.iloc[i:(i + seq_length), 1]
        # Define target
        y = df.iloc[i+seq_length, 1]
        xs.append(x)
        ys.append(y)
    return np.array(xs), np.array(ys)

- Sequential Dataset

In [18]:
import torch
from torch.utils.data import TensorDataset
train_data = df
# Use create_sequences to create inputs and targets
X_train, y_train = create_sequences(train_data, seq_length=24*4)
print(X_train.shape, y_train.shape)

# Create TensorDataset
dataset_train = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train).float(),
)
print(len(dataset_train))

(105119, 96) (105119,)
105119


### Recurrent Neural Networks
Recurrent neuron
- Feed-forward networks
- RNNs: have connections pointing back
- Recurrent neuron:
    - Input `x`
    - Output `y`
    - Hidden state `h`
- In PyTorch: `nn.RNN()`

### Sequence-to-sequence architecture
- Pass sequence as input, use the entire output sequence
- Example: Real-time speech recognition
### Sequence-to-vector architecture
- Pass sequence as input, use only the last output
- Example: Text topic classification
### Vector-to-sequence architecture
- Pass single input, use the entire output sequence
- Example: Text generation
### Encoder-decoder architecture
- Pass entire input sequence, only then start using output sequence
- Example: Machine translation

### RNN In PyTorch
- Sequence-to-vector
    - Define model class with `__init__` method
    - Define recurrent layer. `self.rnn`
    - Define linear layers, `fc`
    - In `forward()`, initialize first hidden state to zeros
    - Pass input and first hidden state throuh RNN layer
    - Select last RNN's output and pass it through linear layer



In [22]:
import torch
import torch.nn as nn
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        h0 = torch.zeros(2, x.size(0), 32)
        out, _ = self.rnn(x, h0)
        out = self.fc(out[:, -1, :])
        return out

Exercise:
- Building a forecasting RNN

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        # Define RNN layer
        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=32,
            num_layers=2,
            batch_first=True,
        )
        self.fc = nn.Linear(32, 1)

    def forward(self, x):
        # Initialize first hidden state with zeros
        h0 = torch.zeros(2, x.size(0), 32)
        # Pass x and h0 through recurrent layer
        out, _ = self.rnn(x, h0)  
        # Pass recurrent layer's last output through linear layer
        out = self.fc(out[:, -1, :])
        return out